# Loan Production Hackathon - Optimized Leak-Safe Pipeline

This notebook keeps the original Google Drive download logic and replaces the modeling flow with a leak-safe, F1-oriented workflow.

In [ ]:
# Optional dependency installer (safe in restricted environments)
import sys
import subprocess
import importlib.util


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        except Exception as e:
            print(f"Could not install {pip_name}: {e}")
    else:
        print(f"{import_name} already available.")

ensure_package("imblearn", "imbalanced-learn")
ensure_package("catboost")
ensure_package("lightgbm")
ensure_package("xgboost")


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
    cross_val_predict,
    learning_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_class_weight

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


## A. Load Training Set (keep original download logic)

In [ ]:
# Read in training data
drive_url = "https://drive.google.com/file/d/1J70Sz3_t7znOFZaDHe3SEtpJ69qCUyZy/view?usp=sharing"

# Convert the Google Drive URL to a direct download URL
file_id = drive_url.split('/d/')[1].split('/')[0]
download_url = f"https://drive.google.com/uc?id={file_id}"

# Read the CSV file into a DataFrame
df_train = pd.read_csv(download_url)

# Display the first few rows of the DataFrame
df_train.head()


In [ ]:
# Inspect training set and required target column
print(df_train.shape)
print("loan_status in train:", "loan_status" in df_train.columns)
df_train.info()


## A. Load Test Set (keep original download logic)

In [ ]:
# Read in test data
drive_url = "https://drive.google.com/file/d/1X7Ezau9dfp1BKYyolYEVZzGQqobtEpGn/view?usp=sharing"

# Convert the Google Drive URL to a direct download URL
file_id = drive_url.split('/d/')[1].split('/')[0]
download_url = f"https://drive.google.com/uc?id={file_id}"

# Read the CSV file into a DataFrame
df_test = pd.read_csv(download_url)

# Display the first few rows of the DataFrame
df_test.head()


In [ ]:
# Inspect test set and verify no target leakage
print(df_test.shape)
print("loan_status in test:", "loan_status" in df_test.columns)
df_test.info()


## B. Build leak-safe preprocessing pipeline

In [ ]:
target_col = "loan_status"
assert target_col in df_train.columns, "Target column loan_status not found in training data."
assert target_col not in df_test.columns, "Test data should not contain loan_status."

X = df_train.drop(columns=[target_col]).copy()
y = df_train[target_col].copy()
X_test = df_test.copy()

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)


## C/D/E. CV evaluation (F1 priority), model comparison, imbalance strategies

In [ ]:
def get_pos_label(y_series):
    unique = sorted(pd.Series(y_series).dropna().unique().tolist())
    if set(unique).issubset({0, 1}):
        return 1
    return unique[-1]


def get_proba_scores(model, X_data):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_data)
        return proba[:, 1] if proba.ndim == 2 else proba
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_data)
        scores = np.asarray(scores)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    return model.predict(X_data)


def evaluate_model_cv(model_name, estimator, X_data, y_data, cv, threshold=0.5):
    fold_rows = []
    fit_times = []
    pos_label = get_pos_label(y_data)
    for fold_id, (tr_idx, va_idx) in enumerate(cv.split(X_data, y_data), start=1):
        X_tr, X_va = X_data.iloc[tr_idx], X_data.iloc[va_idx]
        y_tr, y_va = y_data.iloc[tr_idx], y_data.iloc[va_idx]

        est = clone(estimator)
        t0 = time.time()
        est.fit(X_tr, y_tr)
        fit_times.append(time.time() - t0)

        va_prob = get_proba_scores(est, X_va)
        va_pred = (va_prob >= threshold).astype(int)

        fold_rows.append(
            {
                "fold": fold_id,
                "f1": f1_score(y_va, va_pred, pos_label=pos_label),
                "precision": precision_score(y_va, va_pred, pos_label=pos_label, zero_division=0),
                "recall": recall_score(y_va, va_pred, pos_label=pos_label, zero_division=0),
                "pr_auc": average_precision_score(y_va, va_prob),
            }
        )

    fold_df = pd.DataFrame(fold_rows)
    summary = {
        "model_name": model_name,
        "f1_mean": fold_df["f1"].mean(),
        "f1_std": fold_df["f1"].std(),
        "pr_auc_mean": fold_df["pr_auc"].mean(),
        "precision_mean": fold_df["precision"].mean(),
        "recall_mean": fold_df["recall"].mean(),
        "fit_time": np.mean(fit_times),
    }
    return summary, fold_df

# base class imbalance ratio
class_counts = pd.Series(y).value_counts()
neg_count = class_counts.min() if len(class_counts) == 2 and class_counts.index.max() == 1 else class_counts.iloc[0]
pos_count = class_counts.max() if len(class_counts) == 2 and class_counts.index.max() == 1 else class_counts.iloc[-1]
if len(class_counts) == 2 and 1 in class_counts.index and 0 in class_counts.index:
    scale_pos_weight = class_counts[0] / max(class_counts[1], 1)
else:
    scale_pos_weight = 1.0
print("scale_pos_weight:", round(scale_pos_weight, 4))


In [ ]:
# Build candidate models
log_reg_balanced = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=RANDOM_STATE,
                n_jobs=None,
            ),
        ),
    ]
)

rf_balanced = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=400,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

# Strong model fallback order: CatBoost -> LightGBM -> XGBoost -> HistGB
strong_model_name = None
strong_estimator = None

try:
    from catboost import CatBoostClassifier

    strong_model_name = "CatBoost"
    strong_estimator = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            (
                "model",
                CatBoostClassifier(
                    random_state=RANDOM_STATE,
                    verbose=0,
                    auto_class_weights="Balanced",
                ),
            ),
        ]
    )
except Exception:
    try:
        from lightgbm import LGBMClassifier

        strong_model_name = "LightGBM"
        strong_estimator = Pipeline(
            steps=[
                ("preprocess", preprocessor),
                (
                    "model",
                    LGBMClassifier(
                        random_state=RANDOM_STATE,
                        n_estimators=500,
                        learning_rate=0.05,
                        subsample=0.9,
                        colsample_bytree=0.9,
                        class_weight="balanced",
                    ),
                ),
            ]
        )
    except Exception:
        try:
            from xgboost import XGBClassifier

            strong_model_name = "XGBoost"
            strong_estimator = Pipeline(
                steps=[
                    ("preprocess", preprocessor),
                    (
                        "model",
                        XGBClassifier(
                            random_state=RANDOM_STATE,
                            n_estimators=600,
                            learning_rate=0.05,
                            max_depth=6,
                            subsample=0.9,
                            colsample_bytree=0.9,
                            eval_metric="logloss",
                            scale_pos_weight=scale_pos_weight,
                        ),
                    ),
                ]
            )
        except Exception:
            strong_model_name = "HistGradientBoosting"
            strong_estimator = Pipeline(
                steps=[
                    ("preprocess", preprocessor),
                    (
                        "model",
                        HistGradientBoostingClassifier(
                            random_state=RANDOM_STATE,
                            learning_rate=0.05,
                            max_iter=600,
                            early_stopping=True,
                            validation_fraction=0.1,
                            n_iter_no_change=20,
                        ),
                    ),
                ]
            )

print("Strong model selected:", strong_model_name)

# SMOTE-based comparison (on Logistic Regression)
log_reg_smote = ImbPipeline(
    steps=[
        ("preprocess", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)

candidates = {
    "LogisticRegression_balanced": log_reg_balanced,
    "RandomForest_balanced": rf_balanced,
    f"{strong_model_name}_weighted": strong_estimator,
    "LogisticRegression_SMOTE": log_reg_smote,
}

results = []
fold_details = {}
for name, est in candidates.items():
    summary, fold_df = evaluate_model_cv(name, est, X, y, CV, threshold=0.5)
    results.append(summary)
    fold_details[name] = fold_df

results_df = pd.DataFrame(results).sort_values("f1_mean", ascending=False).reset_index(drop=True)
print("\nModel comparison table:")
display(results_df[["model_name", "f1_mean", "f1_std", "pr_auc_mean", "precision_mean", "recall_mean", "fit_time"]])

best_name = results_df.loc[0, "model_name"]
best_base_estimator = candidates[best_name]
print("Best baseline by CV F1:", best_name)


### 不平衡业务解释（FN/FP）
- **FN（坏客户预测成好客户）**：会带来更高违约风险与资金损失，通常业务代价更高。  
- **FP（好客户预测成坏客户）**：会错杀优质客户，影响放款规模与客户体验。  
- 因此我们用 **F1 + 阈值优化** 平衡 Precision/Recall，并比较 `class_weight/scale_pos_weight` 与 `SMOTE` 两类方案。

## F. RandomizedSearchCV tuning on the selected strong model family

In [ ]:
# Choose strong family estimator for tuning (preferred for final score)
strong_for_tuning = candidates.get(f"{strong_model_name}_weighted", strong_estimator)

param_distributions = {}
if strong_model_name == "CatBoost":
    param_distributions = {
        "model__depth": [4, 5, 6, 7, 8, 9],
        "model__learning_rate": np.linspace(0.01, 0.2, 20),
        "model__l2_leaf_reg": [1, 3, 5, 7, 9, 11],
        "model__iterations": [300, 500, 800, 1000],
    }
elif strong_model_name == "LightGBM":
    param_distributions = {
        "model__num_leaves": [31, 63, 127, 255],
        "model__max_depth": [-1, 4, 6, 8, 10],
        "model__learning_rate": np.linspace(0.01, 0.2, 20),
        "model__n_estimators": [300, 500, 800, 1000],
        "model__min_child_samples": [10, 20, 40, 80],
        "model__subsample": [0.7, 0.8, 0.9, 1.0],
        "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    }
elif strong_model_name == "XGBoost":
    param_distributions = {
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__learning_rate": np.linspace(0.01, 0.2, 20),
        "model__n_estimators": [300, 500, 800, 1000],
        "model__min_child_weight": [1, 3, 5, 7],
        "model__subsample": [0.7, 0.8, 0.9, 1.0],
        "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "model__gamma": [0, 0.1, 0.3, 0.5],
    }
else:
    param_distributions = {
        "model__learning_rate": np.linspace(0.01, 0.2, 20),
        "model__max_iter": [200, 400, 600, 800, 1000],
        "model__max_leaf_nodes": [15, 31, 63, 127],
        "model__min_samples_leaf": [10, 20, 30, 50],
        "model__l2_regularization": [0.0, 0.01, 0.1, 1.0],
    }

search = RandomizedSearchCV(
    estimator=strong_for_tuning,
    param_distributions=param_distributions,
    n_iter=40,
    scoring="f1",
    n_jobs=-1,
    cv=CV,
    random_state=RANDOM_STATE,
    verbose=1,
)

search.fit(X, y)
print("Best params:")
print(search.best_params_)
print("Best CV F1 from RandomizedSearchCV:", search.best_score_)

pre_tune_f1 = results_df.loc[results_df["model_name"] == f"{strong_model_name}_weighted", "f1_mean"].values
pre_tune_f1 = pre_tune_f1[0] if len(pre_tune_f1) else np.nan
print(f"F1 before tuning ({strong_model_name}_weighted): {pre_tune_f1:.5f}")
print(f"F1 after tuning ({strong_model_name}): {search.best_score_:.5f}")


## G. Threshold search with out-of-fold probabilities (core for F1)

In [ ]:
best_tuned_estimator = search.best_estimator_

oof_proba = cross_val_predict(
    best_tuned_estimator,
    X,
    y,
    cv=CV,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

thresholds = np.arange(0.05, 0.951, 0.01)
f1_scores = [f1_score(y, (oof_proba >= t).astype(int)) for t in thresholds]
best_idx = int(np.argmax(f1_scores))
best_threshold = float(thresholds[best_idx])
best_oof_f1 = float(f1_scores[best_idx])

print(f"Best threshold: {best_threshold:.2f}")
print(f"OOF F1 at best threshold: {best_oof_f1:.5f}")

# F1-threshold curve
plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores)
plt.axvline(best_threshold, color="red", linestyle="--", label=f"best={best_threshold:.2f}")
plt.title("F1 vs Threshold (OOF)")
plt.xlabel("Threshold")
plt.ylabel("F1")
plt.legend()
plt.grid(True)
plt.show()

# Precision-Recall curve
precisions, recalls, pr_thresholds = precision_recall_curve(y, oof_proba)
plt.figure(figsize=(6, 5))
plt.plot(recalls, precisions)
plt.title("Precision-Recall Curve (OOF)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.show()

# Confusion matrix + classification report at best threshold
oof_pred_best = (oof_proba >= best_threshold).astype(int)
cm = confusion_matrix(y, oof_pred_best)
print("Confusion matrix:\n", cm)
print("\nClassification report:\n", classification_report(y, oof_pred_best, digits=4))


### 为什么阈值不固定为 0.5
当类别不平衡且业务成本非对称时，0.5 往往不是 F1 最优点。通过 OOF 概率扫描阈值，我们可以在 Precision 与 Recall 之间找到更适合业务目标（降低 FN 或控制 FP）的平衡点。

## H. Learning curve (train vs validation F1)

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_tuned_estimator,
    X,
    y,
    cv=CV,
    scoring="f1",
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 6),
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(8, 4))
plt.plot(train_sizes, train_mean, "o-", label="Train F1")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)
plt.plot(train_sizes, val_mean, "o-", label="Validation F1")
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2)
plt.title("Learning Curve (F1)")
plt.xlabel("Training examples")
plt.ylabel("F1")
plt.legend()
plt.grid(True)
plt.show()


## I. Train final model on full data and generate submit.csv

In [ ]:
# Fit tuned model on full training data
best_tuned_estimator.fit(X, y)

test_proba = best_tuned_estimator.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

# Build submission with strict alignment to sample_submission if available
import os
sample_path = "sample_submission.csv"

if os.path.exists(sample_path):
    sample_submission = pd.read_csv(sample_path)
    submission_df = sample_submission.copy()
    target_like_cols = [c for c in submission_df.columns if c.lower() in ["loan_status", "target", "prediction", "label"]]
    if len(target_like_cols) == 0:
        # fallback to last column as prediction column
        target_pred_col = submission_df.columns[-1]
    else:
        target_pred_col = target_like_cols[0]
    submission_df[target_pred_col] = test_pred
    submission_df = submission_df[sample_submission.columns]
else:
    submission_df = pd.DataFrame({"loan_status": test_pred})

submission_df.to_csv("submit.csv", index=False)

print("submit.csv saved successfully")
print("Best model:", strong_model_name)
print(f"CV F1 mean/std (from tuning): {search.best_score_:.5f} / {results_df['f1_std'].iloc[0]:.5f}")
print(f"Best threshold: {best_threshold:.2f}\n")
print(submission_df.head())
print("submit.csv shape:", submission_df.shape)
